In [ ]:
import torch
print(torch.cuda.is_available())  # 检查 GPU 是否可用
print(torch.cuda.device_count())  # 获取 GPU 数量
print(torch.cuda.get_device_name(0))  # 获取 GPU 0 的名称\





In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import pickle
import dill
# 激活 R 到 Pandas 的转换
pandas2ri.activate()

# 加载 .RData 文件
ro.r['load'](r'/home/zxl/hdd/cellfate/data/list_cell1.RData')



# 提取 R 中的数据框
gene_list = ro.r['gene_list']
B_cell_connectivity_matrices = list(ro.r['B_cell_connectivity_matrices'])
CD4_cell_Tfh_connectivity_matrices = ro.r['CD4_cell_Tfh_connectivity_matrices']
CD4_cell_Th1_connectivity_matrices = ro.r['CD4_cell_Th1_connectivity_matrices']
CD4_cell_Th17_connectivity_matrices = ro.r['CD4_cell_Th17_connectivity_matrices']
CD4_cell_Th2_connectivity_matrices = ro.r['CD4_cell_Th2_connectivity_matrices']
CD4_cell_Treg_connectivity_matrices = ro.r['CD4_cell_Treg_connectivity_matrices']
CD8_cell_Tcm_connectivity_matrices = ro.r['CD8_cell_Tcm_connectivity_matrices']
CD8_cell_Tem_connectivity_matrices = ro.r['CD8_cell_Tem_connectivity_matrices']
B_cell_connectivity_matrices = [pandas2ri.rpy2py(df) for df in B_cell_connectivity_matrices]
CD4_cell_Tfh_connectivity_matrices = [pandas2ri.rpy2py(df) for df in CD4_cell_Tfh_connectivity_matrices]
CD4_cell_Th1_connectivity_matrices = [pandas2ri.rpy2py(df) for df in CD4_cell_Th1_connectivity_matrices]
CD4_cell_Th17_connectivity_matrices = [pandas2ri.rpy2py(df) for df in CD4_cell_Th17_connectivity_matrices]
CD4_cell_Th2_connectivity_matrices = [pandas2ri.rpy2py(df) for df in CD4_cell_Th2_connectivity_matrices]
CD4_cell_Treg_connectivity_matrices = [pandas2ri.rpy2py(df) for df in CD4_cell_Treg_connectivity_matrices]
CD8_cell_Tcm_connectivity_matrices = [pandas2ri.rpy2py(df) for df in CD8_cell_Tcm_connectivity_matrices]
CD8_cell_Tem_connectivity_matrices = [pandas2ri.rpy2py(df) for df in CD8_cell_Tem_connectivity_matrices]

variables_to_save = { 'gene_list': gene_list, 'B_cell_connectivity_matrices': B_cell_connectivity_matrices, 
                    'CD4_cell_Tfh_connectivity_matrices':CD4_cell_Tfh_connectivity_matrices,
                     'CD4_cell_Th1_connectivity_matrices': CD4_cell_Th1_connectivity_matrices, 
                     'CD4_cell_Th17_connectivity_matrices': CD4_cell_Th17_connectivity_matrices, 
                     "CD4_cell_Th2_connectivity_matrices":CD4_cell_Th2_connectivity_matrices,
                     'CD4_cell_Treg_connectivity_matrices':CD4_cell_Treg_connectivity_matrices,
                     'CD8_cell_Tcm_connectivity_matrices': CD8_cell_Tcm_connectivity_matrices, 
                     'CD8_cell_Tem_connectivity_matrices': CD8_cell_Tem_connectivity_matrices
                     }
filename = '/home/zxl/hdd/cellfate/data/Gene_and_network1.pkl'

                
with open(filename, 'wb') as f:
    dill.dump(variables_to_save, f)


In [ ]:
from torch import nn
import shap
import torch
import pandas as pd
import dill
import pickle
import numpy as np
from binn import BINN
from sklearn import preprocessing
import os
test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/result.csv")
dfs=[]
df_group = test_input_data  # 转置为 (样本数, 特征数)
dfs.append(df_group)
X = pd.concat(dfs).fillna(0).to_numpy()
combined_output = torch.Tensor(X).to("cuda:1")
cell_final_layer_name = pd.read_csv("/home/zxl/hdd/cellfate/cell_final_layer_name.csv")
model = torch.load('/home/zxl/hdd/cellfate/model4/binn_model0_tpm_0.001_32.pth', map_location="cuda:1", weights_only=False)
model = model.to("cuda:1")
model.eval()
shap_dict_cell = {"features": [], "shap_values": []}
for name, layer in model.final_layers.named_children():
    if isinstance(layer, nn.Linear):
        explainer = shap.DeepExplainer((model.final_layers, layer), combined_output)
        shap_values = explainer.shap_values(combined_output, check_additivity=False)
                    
        shap_dict_cell["features"].append(cell_final_layer_name)
        shap_dict_cell["shap_values"].append(shap_values)

                 
                   
        combined_output = layer(combined_output)

    elif isinstance(layer, (nn.Tanh, nn.ReLU, nn.LeakyReLU)):
                    
        combined_output = layer(combined_output)

feature_dict_cell = {
                    "name": [],
                    "value": [],
                    "type": [],
             
            }
for sv, features in zip(
                        shap_dict_cell["shap_values"], shap_dict_cell["features"]
                ):
    sv = np.asarray(sv)   
    sv = abs(sv)                                     
    sv_mean1 = np.mean(sv, axis=0)                   
    sv_mean_final1 = sv_mean1
            
    for feature in range(sv_mean1.shape[0]):

        n_classes = sv_mean_final1.shape[1]
    
        for curr_class in range(n_classes):
            feature_dict_cell["name"].append(features[feature])
            feature_dict_cell["value"].append(sv_mean_final1[feature][curr_class])
            feature_dict_cell["type"].append(curr_class)

    shap_df = pd.DataFrame(data=feature_dict_cell)
    shap_csv_path = os.path.join("/home/zxl/hdd/cellfate/model3/", f"shap_cell_iter.csv")
    shap_df.to_csv(shap_csv_path, index=False)
    print(f"SHAP dictionary saved to {shap_csv_path}")

In [2]:
import pandas as pd
test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_100.csv")

In [1]:
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from docs.util_for_examples import fit_data_matrix_to_network_input
test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_100.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_100.csv")
test_pathways = pd.read_csv("/home/zxl/hdd/cellfate/data/BP_5.csv")
test_translation = pd.read_csv("/home/zxl/hdd/cellfate/data/BP_Gene_new.csv")

import pickle
with open('/home/zxl/hdd/cellfate/data/Gene_and_network1.pkl', 'rb') as file:
    data = pickle.load(file)
gene_list = data["gene_list"]

gene_set = set(gene_list.tolist())  # 先用 tolist() 转换为 Python 原生列表
network = Network(
    input_data=test_input_data,
    pathways=test_pathways,
    mapping=test_translation,
    input_data_column="Gene",
    source_column="source",
    target_column="target"
)


binn = BINN(
    network=network,
    activation = "tanh",
    activation_final = "sigmoid",
    connectivity_matrices_list = data,
    dropout=0.2,
    validate=False,
    device="cuda:1",
    learning_rate=0.001,
) 


from binn.explainer import BINNExplainer
from binn.based_cell_train import based_cell_train
explainer = BINNExplainer(binn)
trainer = based_cell_train(binn,explainer)
return_dict= trainer.fit(test_input_data,
                        test_input_sign,
                        nr_iterations=1,
                        connectivity_matrices_list = data,
                        batch_size=32,
                        n_folds=3,
                        val_size = 0.2,
                        test_size = 0.2,
                        max_epochs=10000,
                        num_workers=20,
                        gene_list=gene_list)




/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zxl/hdd/cellfate/binn/binn.py:214: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.loss = nn.CrossEntropyLoss(weight=torch.tensor(self.weight, device=device))



BINN is on the device: cuda:1
BINN(
  (batchnorm_np): BatchNorm1d(7057, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (B_cell_layers): Sequential(
    (Layer_0): Linear(in_features=7057, out_features=1044, bias=True)
    (BatchNorm_0): BatchNorm1d(1044, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (Dropout_0): Dropout(p=0.2, inplace=False)
    (Tanh 0): Tanh()
    (Layer_1): Linear(in_features=1044, out_features=522, bias=True)
    (BatchNorm_1): BatchNorm1d(522, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (Dropout_1): Dropout(p=0.2, inplace=False)
    (Tanh 1): Tanh()
    (Layer_2): Linear(in_features=522, out_features=121, bias=True)
    (BatchNorm_2): BatchNorm1d(121, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (Dropout_2): Dropout(p=0.2, inplace=False)
    (Tanh 2): Tanh()
    (Layer_3): Linear(in_features=121, out_features=16, bias=True)
    (BatchNorm_3): BatchNorm1d(16, eps=1e-05, momentu

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



BINN is on the device: cuda:1


You are using a CUDA device ('NVIDIA GeForce RTX 5090 D') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
2025-07-04 09:59:49.446567: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` a

Optimized Temperature: 1.2758


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=77` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_acc          │    0.9599999785423279     │
│         val_loss          │    0.22387896478176117    │
└───────────────────────────┴───────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=77` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test_F1          │    0.9659786224365234     │
│         test_acc          │    0.9692753553390503     │
└───────────────────────────┴───────────────────────────┘

In [ ]:
import pandas as pd
data=return_dict
metrics = {
        'iteration': data['iteration'],
        'train_acc': [x[0] for x in data['train_acc']],
        'train_loss': [x[0] for x in data['train_loss']],
        'val_acc': [x[0] for x in data['val_acc']],
        'val_loss': [x[0] for x in data['val_loss']],
        'test_acc': [x[0] for x in data['test_acc']],
        'test_f1': [x[0] for x in data['test_f1']],
        'epochs': [x[0] for x in data['epochs']]
    }
    
df = pd.DataFrame(metrics)
    
    # 保存为CSV\n",
df.to_csv("/home/zxl/hdd/cellfate/model4/metrics_0.001_32_删去unT细胞.csv", index=False)

In [1]:
import torch
import pandas as pd
import numpy as np
from torch import nn as nn
import pickle
from sklearn.metrics import precision_score
from torchmetrics.classification import MulticlassF1Score
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score
# Device configuration
device = torch.device("cpu")

def calculate_accuracy(y, prediction) -> float:
    return torch.sum(y == prediction).item() / float(len(y))

def calculate_F1(y, prediction, num_classes, device) -> float:
    metric = MulticlassF1Score(num_classes=num_classes).to(device)
    return metric(prediction, y)

# Load model and data
model = torch.load('/home/zxl/hdd/cellfate/model4/binn_model9_tpm_0.001_32.pth', map_location=device, weights_only=False)
model = model.to(device)
model.eval()
#print(model)
with open('/home/zxl/hdd/cellfate/model4/test9_tpm_0.001_32.pkl', 'rb') as file:
    data = pickle.load(file)

# Prepare data - ensure everything is on the same device
X_test = torch.Tensor(data["X_test"]).to(device)
y_test = torch.LongTensor(data["y_test"]).to(device)  # Use LongTensor for classification labels

# Make predictions
with torch.no_grad():
    predictions = model(X_test)
    predicted_labels = torch.argmax(predictions, dim=1)

def predict_proba(model, X):
    with torch.no_grad():
        logits = model(X)
        probabilities = torch.softmax(logits, dim=1)
        return probabilities

# Calculate metrics

print('acc:', calculate_accuracy(y_test, predicted_labels))
print('f1_score', f1_score(predicted_labels, y_test,average='weighted'))    
print('precision_score', precision_score(predicted_labels, y_test, average='weighted', zero_division=0))
print('recall_score', recall_score(predicted_labels, y_test, average='weighted', zero_division=0))

# # Move tensors to CPU for sklearn metrics
# y_np = y_test.cpu().numpy()
# predicted_labels_np = predicted_labels.cpu().numpy()
# macro_precision = precision_score(y_np, predicted_labels_np, average='macro')

# print(f"acc: {acc}, f1: {f1}, macro_precision: {macro_precision}")

# # Get probabilities
# proba = predict_proba(model, X_test)

# # Save probabilities to CSV
# proba_df = pd.DataFrame(proba.cpu().numpy())
# proba_df.to_csv('/home/zxl/hdd/cellfate/result/predict_proba.csv', index=False)
# print("Probabilities saved to CSV")


/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


acc: 0.9715942028985507
f1_score 0.9714901498398825
precision_score 0.9720850190972637
recall_score 0.9715942028985507


In [1]:
import torch
import pandas as pd
from torch import nn as nn
import numpy as np
import pickle
import collections
from sklearn.metrics import precision_score
from torchmetrics.classification import MulticlassF1Score
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score
class SimplifiedBINN(nn.Module):

    def __init__(self):
        super().__init__()
        # 仅保留训练时实际使用的层
        def _generate_final(final_cat: int , bias=False, n_outputs = 24):
            layers = []   

            layers.append(("BatchNorm_{n}", nn.BatchNorm1d(final_cat)))
            layers.append(("Dropout", nn.Dropout(0.2)))
            layers.append(
            (
                "final",
                nn.Linear(final_cat, n_outputs, bias=bias),
            )
            )
    

            layers.append(("identity_final",nn.Identity()))
            model = nn.Sequential(collections.OrderedDict(layers))

            return(model)
        self.B_cell_layers = model.B_cell_layers
        self.final_layers= _generate_final(96,  bias=False, n_outputs = 24)
    
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        outputs=[]
        
        self.cell_layers = self.B_cell_layers
        outputs.append(model._forward_residual(x))
        x = self.final_layers(torch.cat(outputs, dim=1))
        return x


original_model = torch.load('/home/zxl/hdd/cellfate/model4/binn_model_unB0_tpm_100_16.pth', map_location=device, weights_only=False)
filtered_state_dict = {
    k: v for k, v in original_model.state_dict().items() 
    if k.startswith(('B_cell_layers', 'final_layers'))
}

# 初始化精简模型并加载参数
simplified_model = SimplifiedBINN()
simplified_model.load_state_dict(filtered_state_dict, strict=False)
simplified_model.eval()
with open('/home/zxl/hdd/cellfate/model4/test__unb0_tpm_100_16.pkl', 'rb') as file:
    data = pickle.load(file)

# Prepare data - ensure everything is on the same device
X_test = torch.Tensor(data["X_test"]).to(device)
y_test = torch.LongTensor(data["y_test"]).to(device)  # Use LongTensor for classification labels
with torch.no_grad():
    output = simplified_model(X_test)
    
    predicted_labels = torch.argmax(output, dim=1)

# Calculate metrics

print('acc:', calculate_accuracy(y_test, predicted_labels))
print('f1_score', f1_score(predicted_labels, y_test,average='weighted'))    
print('precision_score', precision_score(predicted_labels, y_test, average='weighted', zero_division=0))
print('recall_score', recall_score(predicted_labels, y_test, average='weighted', zero_division=0))
# 预测（使用与训练时相同维度的输入）

# def predict_and_evaluate(model, X, y_true):
#     with torch.no_grad():
#         output = simplified_model(test_data)
#         probabilities = torch.softmax(output, dim=1)
            
#             # 提取预测类别和对应的概率
#         pred_probs, pred_labels = torch.max(probabilities, dim=1)
            
#             # 转换为numpy数组
#         pred_probs = pred_probs.cpu().numpy()
#         pred_labels = pred_labels.cpu().numpy()
#         y_true_np = y_true.cpu().numpy()
            
#             # 生成表格
#         df = pd.DataFrame({
#                 "预测概率": pred_probs,
#                 "预测是否正确": (pred_labels == y_true_np).astype(int),
#                 "哪一类预测正确": y_true_np,               # 显示真实类别
#                 "预测类别":pred_labels 
#             })
#     print('acc:', calculate_accuracy(y_test, predicted_labels))
#     print('f1_score', f1_score(predicted_labels, y_test,average='weighted'))    
#     print('precision_score', precision_score(predicted_labels, y_test, average='weighted', zero_division=0))
#     print('recall_score', recall_score(predicted_labels, y_test, average='weighted', zero_division=0))
#     return df

NameError: name 'device' is not defined

In [1]:
import torch
import pandas as pd
import pickle
import numpy as np
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score

device = torch.device("cpu")

# ------------------------ 工具函数定义 ------------------------
def predict_and_evaluate(model, X, y_true):
    """
    生成预测概率和预测结果的表格
    参数:
        model: 训练好的PyTorch模型
        X: 输入数据 (Tensor)
        y_true: 真实标签 (Tensor)
    返回:
        DataFrame: 包含预测结果的表格
    """
    with torch.no_grad():
        probabilities = torch.softmax(model(X), dim=1)
        pred_probs, pred_labels = torch.max(probabilities, dim=1)
        
        # 转换为CPU numpy数组
        pred_probs = pred_probs.cpu().numpy()
        pred_labels = pred_labels.cpu().numpy()
        y_true_np = y_true.cpu().numpy()  # 确保y_true是Tensor
        
        df = pd.DataFrame({
            "预测概率": pred_probs,
            "预测是否正确": (pred_labels == y_true_np).astype(int),
            "真实类别": y_true_np,
            "预测类别": pred_labels
        })
    return df

def _fit_data_matrix_to_network_input(data_matrix, features, feature_column="Gene"):
    """数据对齐预处理"""
    if len(features) > len(data_matrix.index):
        features_df = pd.DataFrame(features, columns=[feature_column])
        data_matrix = data_matrix.merge(features_df, how="right", on=feature_column)
    if len(features) > 0:
        data_matrix.set_index(feature_column, inplace=True)
        data_matrix = data_matrix.loc[features]
    return data_matrix

def _generate_k_folds(data_matrix, design_matrix, groups, n_folds=3, test_size=0.1, random_state=42):
    """生成交叉验证数据"""
    y_list, dfs, sample_names = [], [], []
    for group in groups:
        group_samples = design_matrix[design_matrix["group"] == group]["sample"].values
        df_group = data_matrix[group_samples].T  # (样本数, 特征数)
        dfs.append(df_group)
        y_list += [group - 1] * len(group_samples)  # 标签从0开始
        sample_names.extend(group_samples)
    
    X = pd.concat(dfs).fillna(0).to_numpy()
    y = np.array(y_list)
    return X, y, sample_names

# ------------------------ 主流程 ------------------------
# 加载模型
model = torch.load('/home/zxl/hdd/cellfate/model4/binn_model_unB0_tpm_100_16.pth', map_location=device, weights_only=False)
model = model.to(device)
model.eval()

# 加载测试数据
with open('/home/zxl/hdd/cellfate/model4/test__unB0_tpm_100_16.pkl', 'rb') as f:
    test_data = pickle.load(f)
X_test = torch.Tensor(test_data["X_test"]).to(device)
y_test = torch.LongTensor(test_data["y_test"]).to(device)  # Use LongTensor for classification labels

# 验证集评估
with torch.no_grad():
    predictions = model(X_test)
    predicted_labels = torch.argmax(predictions, dim=1)

# 计算指标
def calculate_accuracy(y_true, y_pred):
    return (y_true == y_pred).float().mean().item()

print('准确率:', calculate_accuracy(y_test, predicted_labels))
print('F1分数:', f1_score(y_test.cpu(), predicted_labels.cpu(), average='weighted'))
print('精确率:', precision_score(y_test.cpu(), predicted_labels.cpu(), average='weighted', zero_division=0))
print('召回率:', recall_score(y_test.cpu(), predicted_labels.cpu(), average='weighted', zero_division=0))
print('acc:', calculate_accuracy(y_test, predicted_labels))
print('f1_score', f1_score(y_test.cpu(),predicted_labels.cpu(), average='weighted'))    
print('precision_score', precision_score(y_test.cpu(),predicted_labels.cpu(),  average='weighted', zero_division=0))
print('recall_score', recall_score(y_test.cpu(),predicted_labels.cpu(),  average='weighted', zero_division=0))
# ------------------------ 对新数据预测 ------------------------
# 加载新数据
input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/MET500_621_exp.csv")
label_df = pd.read_csv("/home/zxl/hdd/cellfate/data/MET500_621_group.csv")  # 注意变量名改为label_df避免冲突

# 数据预处理
fitted_input = _fit_data_matrix_to_network_input(
    input_data.reset_index(), 
    features=test_data["features"], 
    feature_column="Gene"
)
X_new, y_new, samples = _generate_k_folds(
    fitted_input, 
    design_matrix=label_df,  # 使用清晰的变量名
    groups=np.arange(1, 25)
)

# 转换为Tensor并预测
X_new_tensor = torch.tensor(X_new, dtype=torch.float32).to(device)
y_new_tensor = torch.tensor(y_new, dtype=torch.long).to(device)

with torch.no_grad():
    new_preds = model(X_new_tensor)
    new_probs, new_labels = torch.max(new_preds, dim=1)

# 生成结果表
result_df = pd.DataFrame({
    "样本名": samples,
    "预测是否正确": (new_labels.cpu().numpy() == y_new).astype(int),
    "真实类别": y_new,
    "预测类别": new_labels.cpu().numpy()
})


result_df.to_csv('/home/zxl/hdd/cellfate/result/prediction_results_MET500.csv', index=False)
# 加载新数据
input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/bcgsc_283_exp.csv")
label_df = pd.read_csv("/home/zxl/hdd/cellfate/data/bcgsc_283_group.csv")  # 注意变量名改为label_df避免冲突

# 数据预处理
fitted_input = _fit_data_matrix_to_network_input(
    input_data.reset_index(), 
    features=test_data["features"], 
    feature_column="Gene"
)
X_new, y_new, samples = _generate_k_folds(
    fitted_input, 
    design_matrix=label_df,  # 使用清晰的变量名
    groups=np.arange(1, 25)
)

# 转换为Tensor并预测
X_new_tensor = torch.tensor(X_new, dtype=torch.float32).to(device)
y_new_tensor = torch.tensor(y_new, dtype=torch.long).to(device)

with torch.no_grad():
    new_preds = model(X_new_tensor)
    new_probs, new_labels = torch.max(new_preds, dim=1)

# 生成结果表
result_df = pd.DataFrame({
    "样本名": samples,
    "预测是否正确": (new_labels.cpu().numpy() == y_new).astype(int),
    "真实类别": y_new,
    "预测类别": new_labels.cpu().numpy()
})


result_df.to_csv('/home/zxl/hdd/cellfate/result/prediction_results_bcgsc.csv', index=False)

input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/ICGC_primary_tpm_100.csv")
label_df = pd.read_csv("/home/zxl/hdd/cellfate/data/ICGCsub_primary_tpm_100.csv")  # 注意变量名改为label_df避免冲突

# 数据预处理
fitted_input = _fit_data_matrix_to_network_input(
    input_data.reset_index(), 
    features=test_data["features"], 
    feature_column="Gene"
)
X_new, y_new, samples = _generate_k_folds(
    fitted_input, 
    design_matrix=label_df,  # 使用清晰的变量名
    groups=np.arange(1, 25)
)

# 转换为Tensor并预测
X_new_tensor = torch.tensor(X_new, dtype=torch.float32).to(device)
y_new_tensor = torch.tensor(y_new, dtype=torch.long).to(device)

with torch.no_grad():
    new_preds = model(X_new_tensor)
    new_probs, new_labels = torch.max(new_preds, dim=1)

# 生成结果表
result_df = pd.DataFrame({
    "样本名": samples,
    "预测是否正确": (new_labels.cpu().numpy() == y_new).astype(int),
    "真实类别": y_new,
    "预测类别": new_labels.cpu().numpy()
})


result_df.to_csv('/home/zxl/hdd/cellfate/result/prediction_results_ICGC.csv', index=False)



input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_1165_exp.csv")
label_df = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_1165_group.csv")  # 注意变量名改为label_df避免冲突

# 数据预处理
fitted_input = _fit_data_matrix_to_network_input(
    input_data.reset_index(), 
    features=test_data["features"], 
    feature_column="Gene"
)
X_new, y_new, samples = _generate_k_folds(
    fitted_input, 
    design_matrix=label_df,  # 使用清晰的变量名
    groups=np.arange(1, 25)
)

# 转换为Tensor并预测
X_new_tensor = torch.tensor(X_new, dtype=torch.float32).to(device)
y_new_tensor = torch.tensor(y_new, dtype=torch.long).to(device)

with torch.no_grad():
    new_preds = model(X_new_tensor)
    new_probs, new_labels = torch.max(new_preds, dim=1)

# 生成结果表
result_df = pd.DataFrame({
    "样本名": samples,
    "预测是否正确": (new_labels.cpu().numpy() == y_new).astype(int),
    "真实类别": y_new,
    "预测类别": new_labels.cpu().numpy()
})


result_df.to_csv('/home/zxl/hdd/cellfate/result/prediction_results_TCGAMET.csv', index=False)

input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/MET500_621_exp.csv")
label_df = pd.read_csv("/home/zxl/hdd/cellfate/data/MET500_621_group.csv")  # 注意变量名改为label_df避免冲突

# 数据预处理
fitted_input = _fit_data_matrix_to_network_input(
    input_data.reset_index(), 
    features=test_data["features"], 
    feature_column="Gene"
)
X_new, y_new, samples = _generate_k_folds(
    fitted_input, 
    design_matrix=label_df,  # 使用清晰的变量名
    groups=np.arange(1, 25)
)

# 转换为Tensor并预测
X_new_tensor = torch.tensor(X_new, dtype=torch.float32).to(device)
y_new_tensor = torch.tensor(y_new, dtype=torch.long).to(device)

with torch.no_grad():
    new_preds = model(X_new_tensor)
    new_probs, new_labels = torch.max(new_preds, dim=1)

# 生成结果表
result_df = pd.DataFrame({
    "样本名": samples,
    "预测是否正确": (new_labels.cpu().numpy() == y_new).astype(int),
    "真实类别": y_new,
    "预测类别": new_labels.cpu().numpy()
})


result_df.to_csv('/home/zxl/hdd/cellfate/result/prediction_results_MET500.csv', index=False)

print("预测结果已保存！")


/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


准确率: 0.9692753553390503
F1分数: 0.9694425185939324
精确率: 0.9702930613511377
召回率: 0.9692753623188406
acc: 0.9692753553390503
f1_score 0.9694425185939324
precision_score 0.9702930613511377
recall_score 0.9692753623188406
预测结果已保存！


In [2]:
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score
import pandas as pd 
import numpy as np
import csv
#prediction_results_TCGA = pd.read_csv('/home/zxl/hdd/cellfate/result/prediction_results_TCGAtest_XGBOOST.csv')
# prediction_results_TCGA["pred"] = prediction_results_TCGA["pred"]-1
# prediction_results_TCGA["V4"] = np.where(
#     prediction_results_TCGA["pred"] == prediction_results_TCGA["group"], 
#     1, 
#     0
# )
prediction_results_TCGA1165 = pd.read_csv('/home/zxl/hdd/cellfate/result/prediction_results_TCGAMET.csv')
prediction_results_bcgsc = pd.read_csv('/home/zxl/hdd/cellfate/result/prediction_results_bcgsc.csv')
prediction_results_ICGC = pd.read_csv('/home/zxl/hdd/cellfate/result/prediction_results_ICGC.csv')
prediction_results_MET500 = pd.read_csv('/home/zxl/hdd/cellfate/result/prediction_results_MET500.csv')
def eval(pred):
    Y_pre_label_change = pred['预测类别'] 
    Y_true_label = pred['真实类别'] 
    ture = pred['预测是否正确']
    print('ACC_score',sum((ture==1))/len(ture))
    print('f1_score', f1_score(Y_pre_label_change, Y_true_label,average='weighted'))    
    print('precision_score', precision_score(Y_pre_label_change, Y_true_label, average='weighted', zero_division=0))
    print('recall_score', recall_score(Y_pre_label_change, Y_true_label, average='weighted', zero_division=0))

    print(sum((ture==1)))
    print(len(ture))


#eval(prediction_results_TCGA)
#print('*'*100)
eval(prediction_results_TCGA1165)
print('*'*100)
eval(prediction_results_bcgsc)
print('*'*100)
eval(prediction_results_ICGC)
print('*'*100)
eval(prediction_results_MET500)

ACC_score 0.976824034334764
f1_score 0.9758020434000766
precision_score 0.9755664852349557
recall_score 0.976824034334764
1138
1165
****************************************************************************************************
ACC_score 0.773851590106007
f1_score 0.7507663460709968
precision_score 0.7375688341931481
recall_score 0.773851590106007
219
283
****************************************************************************************************
ACC_score 0.9569560047562425
f1_score 0.9508840647637531
precision_score 0.9474283551716656
recall_score 0.9569560047562425
8048
8410
****************************************************************************************************
ACC_score 0.6634460547504025
f1_score 0.6003122201956627
precision_score 0.6237764947166565
recall_score 0.6634460547504025
412
621


In [ ]:
import torch
import pandas as pd
import pickle
import numpy as np
from sklearn.metrics import precision_score

def predict_proba(model, X):
    X = torch.from_numpy(X).float()
    with torch.no_grad():

        logits = model(X)
        probabilities = torch.softmax(logits, dim=1)
        return probabilities
def _fit_data_matrix_to_network_input(
    data_matrix: pd.DataFrame, feature, feature_column="Gene", 
        ) -> pd.DataFrame:
        nr_features_in_matrix = len(data_matrix.index)
        if len(feature) > nr_features_in_matrix:
            features_df = pd.DataFrame(feature, columns=[feature_column])
            data_matrix = data_matrix.merge(features_df, how="right", on=feature_column)
        if len(feature) > 0:
            data_matrix.set_index(feature_column, inplace=True)
            feature = list(feature)
      
            data_matrix = data_matrix.loc[feature]
        return data_matrix
def _generate_k_folds(
        data_matrix: pd.DataFrame,
        design_matrix: pd.DataFrame
):
 
    dfs = []
    sample_names = []  # 新增：用于记录样本名字

    group_samples = design_matrix["sample"].values
    df_group = data_matrix[group_samples].T  # 转置为 (样本数, 特征数)
    dfs.append(df_group)
    sample_names.extend(group_samples)  # 新增：记录样本名字
    X = pd.concat(dfs).fillna(0).to_numpy()
    # X = preprocessing.StandardScaler().fit_transform(X)
    return X,  sample_names  # 新增：返回样本名字
# 示例用法
if __name__ == "__main__":
    model = torch.load('/home/zxl/hdd/cellfate/model4/binn_model0_tpm_0.001_32.pth', weights_only=False)
    with open('/home/zxl/hdd/cellfate/model4/test0_tpm_0.001_32.pkl', 'rb') as file:
        data = pickle.load(file)
    input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/GSE2109_exp.csv")
    features = data["features"]
    y_true = pd.read_csv("/home/zxl/hdd/cellfate/data/GSE2109_group.csv")
    fitted_input_data = _fit_data_matrix_to_network_input(
                input_data.reset_index(),
                
                feature_column="Gene",
                feature = features
            )
    X,  sample_names = _generate_k_folds(
                fitted_input_data,
                design_matrix=y_true
            )
    test_data = torch.Tensor(X).to("cpu")

    proba = predict_proba(model, X)

# Save probabilities to CSV
    proba_df = pd.DataFrame(proba.cpu().numpy())
    proba_df.insert(0, "样本名字", sample_names)  # 新增：在 DataFrame 中插入样本名字列
    proba_df.to_csv('/home/zxl/hdd/cellfate/result/predict_proba_GSE2109.csv', index=False)

/home/zxl/.conda/envs/cellfate/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from sklearn.metrics import precision_score
def calculate_accuracy(y, prediction) -> float:
    return torch.sum(y == prediction).item() / float(1725)
model = torch.load('/home/zxl/hdd/cellfate/model/binn_model1_tpm_100.pth').cuda()
with open('/home/zxl/hdd/cellfate/model/test1_tpm_100.pkl', 'rb') as file:
    data = pickle.load(file)
X_test = data["X_test"]
y_test = data["y_test"]
X_test = torch.Tensor(X_test).cuda()
model.eval()
def predict_proba(model, X):
    with torch.no_grad():
        logits = model(X)
        probabilities = torch.softmax(logits, dim=1)
        return probabilities


def calculate_confidence_acc(model, X, y_true, confidence_threshold):
    """
    计算置信度大于指定阈值的样本的准确率
    :param model: 模型
    :param X: 输入数据
    :param y_true: 真实标签
    :param confidence_threshold: 置信度阈值
    :return: 符合条件的样本的准确率
    """
    # 获取预测概率
    probabilities = predict_proba(model, X)
    
    # 获取最大置信度及其对应的预测标签
    max_confidences, predicted_labels = torch.max(probabilities, dim=1)
    
    # 筛选置信度大于阈值的样本
    mask = max_confidences <= confidence_threshold
    filtered_y_true = y_true[mask]
    filtered_y_pred = predicted_labels[mask]
    
    # 如果没有样本满足条件，返回0
    if len(filtered_y_true) == 0:
        return 0.0
    
    # 计算准确率
    return calculate_accuracy(filtered_y_true, filtered_y_pred)

# 示例用法
# 假设 model 是训练好的模型，X_test 是测试数据，y_test 是测试标签
y_test = torch.Tensor(y_test).cuda()  # 将 y_test 转换为 CUDA 张量

# 计算不同置信度下的准确率
confidence_thresholds = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
for threshold in confidence_thresholds:
    acc = calculate_confidence_acc(model, X_test, y_test, threshold)
    print(f"Confidence > {threshold}: ACC = {acc:.4f}")

In [ ]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.eval import evaluate
device = torch.device("cpu")
with open('/home/zxl/hdd/cellfate/model4/test0_tpm_0.001_32.pkl', 'rb') as file:
    data = pickle.load(file)
features = data["features"]
icgc_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/ICGC_primary_tpm_100.csv")
icgc_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/ICGCsub_primary_tpm_100.csv")

model = torch.load('/home/zxl/hdd/cellfate/model4/binn_model0_tpm_0.001_32.pth', map_location=device, weights_only=False)
model = model.to(device)
eval_sign = evaluate(input_data= icgc_input_data,design_matrix= icgc_input_sign,feature = features, model = model)




In [1]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_441.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_441.csv")

import gc


for i in range(1, 5):
    # 确保从干净状态开始
    print(i)
    torch.cuda.empty_cache()  
    gc.collect()

    try:
        # 加载模型到 GPU

        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") 
                # 加载模型到 GPU
        model_path = f'/home/zxl/hdd/cellfate/model4/binn_model{i}_tpm_0.001_32.pth'
        model = torch.load(model_path, map_location=device,weights_only=False)
        
                # SHAP 解释过程
        shap = SHAPExplainer(
                input_data=test_input_data, 
                design_matrix=test_input_sign, 
                model=model,
                device=device
                )

        shap.explain(
                output_dir="/home/zxl/hdd/cellfate/model3", 
                iteration=i
        

                )
        print(f"Completed iteration {i}")

    finally:
        # 显式释放资源
        if 'model' in locals():
            model.cpu()          # 将模型移出 GPU
            del model            # 删除模型引用
        if 'shap' in locals():
            del shap             # 删除 SHAP 解释器引用
        
        # 强制清理显存
        torch.cuda.empty_cache()
        gc.collect()

/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1
SHAPExplainer initialized.
当前时间: 2025-06-02 20:03:08.997317
[tensor([[ 0.1008,  0.0367,  0.7936,  ...,  0.6927, -0.8293, -0.3933],
        [-0.3147,  0.2833,  0.7986,  ...,  0.7638, -0.8293, -0.4851],
        [ 0.7392, -0.4190,  0.8566,  ...,  0.8458, -0.9276, -0.4966],
        ...,
        [ 0.7496,  0.8866,  0.7957,  ...,  0.6719, -0.8191, -0.6337],
        [ 0.9339,  0.6854,  0.3458,  ...,  0.8399, -0.7768, -0.6769],
        [ 0.4201,  0.8752,  0.3989,  ...,  0.5266, -0.7047, -0.5746]],
       device='cuda:0'), tensor([[-0.6412,  0.7651, -0.6716,  ..., -0.5391, -0.2205, -0.8679],
        [-0.1225,  0.1276, -0.2649,  ..., -0.5788, -0.2473, -0.8747],
        [ 0.1550,  0.6731, -0.7194,  ..., -0.5658, -0.2683, -0.8830],
        ...,
        [-0.4825, -0.1516,  0.6766,  ..., -0.6113,  0.3710, -0.7309],
        [-0.8622,  0.4480,  0.8420,  ..., -0.6430,  0.0855, -0.8655],
        [-0.5491,  0.5227,  0.7933,  ..., -0.2906,  0.7483, -0.8273]],
       device='cuda:0'), tensor([[ 0.8827, -

/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/shap/explainers/_deep/deep_pytorch.py:243: UserWarning: unrecognized nn.Module: Identity
  warnings.warn(f'unrecognized nn.Module: {module_type}')


当前时间: 2025-06-02 22:15:02.839925

Prefix 'Pro_B_cell_layers' found at indices: [36, 37, 38, 39]
Prefix 'Pre_B_cell_layers' found at indices: [40, 41, 42, 43]
Prefix 'naive_CD4_cell_Tfh_layers' found at indices: [152, 153, 154, 155]
Prefix 'CLP_CD4_cell_Tfh_layers' found at indices: [140, 141, 142, 143]
Prefix 'naive_CD4_cell_Th17_layers' found at indices: [136, 137, 138, 139]
Prefix 'mature_B_cell_layers' found at indices: [48, 49, 50, 51]
Prefix 'naive_CD8_cell_Tcm_layers' found at indices: [64, 65, 66, 67]
Prefix 'HSC_CD4_cell_Tfh_layers' found at indices: [24, 25, 26, 27]
Prefix 'Pre_CD4_cell_Th1_layers' found at indices: [100, 101, 102, 103]
Prefix 'CLP_CD4_cell_Th2_layers' found at indices: [108, 109, 110, 111]
Prefix 'Teff_CD8_cell_Tem_layers' found at indices: [88, 89, 90, 91]
Prefix 'naive_CD8_cell_Tem_layers' found at indices: [84, 85, 86, 87]
Prefix 'CLP_CD8_cell_Tem_layers' found at indices: [72, 73, 74, 75]
Prefix 'HSC_CD4_cell_Treg_layers' found at indices: [28, 29, 30, 31

/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/shap/explainers/_deep/deep_pytorch.py:243: UserWarning: unrecognized nn.Module: Identity
  warnings.warn(f'unrecognized nn.Module: {module_type}')


当前时间: 2025-06-03 00:41:18.885412

Prefix 'Pro_B_cell_layers' found at indices: [36, 37, 38, 39]
Prefix 'Pre_B_cell_layers' found at indices: [40, 41, 42, 43]
Prefix 'naive_CD4_cell_Tfh_layers' found at indices: [152, 153, 154, 155]
Prefix 'CLP_CD4_cell_Tfh_layers' found at indices: [140, 141, 142, 143]
Prefix 'naive_CD4_cell_Th17_layers' found at indices: [136, 137, 138, 139]
Prefix 'mature_B_cell_layers' found at indices: [48, 49, 50, 51]
Prefix 'naive_CD8_cell_Tcm_layers' found at indices: [64, 65, 66, 67]
Prefix 'HSC_CD4_cell_Tfh_layers' found at indices: [24, 25, 26, 27]
Prefix 'Pre_CD4_cell_Th1_layers' found at indices: [100, 101, 102, 103]
Prefix 'CLP_CD4_cell_Th2_layers' found at indices: [108, 109, 110, 111]
Prefix 'Teff_CD8_cell_Tem_layers' found at indices: [88, 89, 90, 91]
Prefix 'naive_CD8_cell_Tem_layers' found at indices: [84, 85, 86, 87]
Prefix 'CLP_CD8_cell_Tem_layers' found at indices: [72, 73, 74, 75]
Prefix 'HSC_CD4_cell_Treg_layers' found at indices: [28, 29, 30, 31

/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/shap/explainers/_deep/deep_pytorch.py:243: UserWarning: unrecognized nn.Module: Identity
  warnings.warn(f'unrecognized nn.Module: {module_type}')


当前时间: 2025-06-03 03:06:34.512567

Prefix 'Pro_B_cell_layers' found at indices: [36, 37, 38, 39]
Prefix 'Pre_B_cell_layers' found at indices: [40, 41, 42, 43]
Prefix 'naive_CD4_cell_Tfh_layers' found at indices: [152, 153, 154, 155]
Prefix 'CLP_CD4_cell_Tfh_layers' found at indices: [140, 141, 142, 143]
Prefix 'naive_CD4_cell_Th17_layers' found at indices: [136, 137, 138, 139]
Prefix 'mature_B_cell_layers' found at indices: [48, 49, 50, 51]
Prefix 'naive_CD8_cell_Tcm_layers' found at indices: [64, 65, 66, 67]
Prefix 'HSC_CD4_cell_Tfh_layers' found at indices: [24, 25, 26, 27]
Prefix 'Pre_CD4_cell_Th1_layers' found at indices: [100, 101, 102, 103]
Prefix 'CLP_CD4_cell_Th2_layers' found at indices: [108, 109, 110, 111]
Prefix 'Teff_CD8_cell_Tem_layers' found at indices: [88, 89, 90, 91]
Prefix 'naive_CD8_cell_Tem_layers' found at indices: [84, 85, 86, 87]
Prefix 'CLP_CD8_cell_Tem_layers' found at indices: [72, 73, 74, 75]
Prefix 'HSC_CD4_cell_Treg_layers' found at indices: [28, 29, 30, 31

/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/shap/explainers/_deep/deep_pytorch.py:243: UserWarning: unrecognized nn.Module: Identity
  warnings.warn(f'unrecognized nn.Module: {module_type}')


当前时间: 2025-06-03 05:30:32.358869

Prefix 'Pro_B_cell_layers' found at indices: [36, 37, 38, 39]
Prefix 'Pre_B_cell_layers' found at indices: [40, 41, 42, 43]
Prefix 'naive_CD4_cell_Tfh_layers' found at indices: [152, 153, 154, 155]
Prefix 'CLP_CD4_cell_Tfh_layers' found at indices: [140, 141, 142, 143]
Prefix 'naive_CD4_cell_Th17_layers' found at indices: [136, 137, 138, 139]
Prefix 'mature_B_cell_layers' found at indices: [48, 49, 50, 51]
Prefix 'naive_CD8_cell_Tcm_layers' found at indices: [64, 65, 66, 67]
Prefix 'HSC_CD4_cell_Tfh_layers' found at indices: [24, 25, 26, 27]
Prefix 'Pre_CD4_cell_Th1_layers' found at indices: [100, 101, 102, 103]
Prefix 'CLP_CD4_cell_Th2_layers' found at indices: [108, 109, 110, 111]
Prefix 'Teff_CD8_cell_Tem_layers' found at indices: [88, 89, 90, 91]
Prefix 'naive_CD8_cell_Tem_layers' found at indices: [84, 85, 86, 87]
Prefix 'CLP_CD8_cell_Tem_layers' found at indices: [72, 73, 74, 75]
Prefix 'HSC_CD4_cell_Treg_layers' found at indices: [28, 29, 30, 31

In [ ]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_100.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_100.csv")

import gc


        # 加载模型到 GPU
model_path = f'/home/zxl/hdd/cellfate/model4/binn_model0_tpm_0.001_32.pth'
model = torch.load(model_path, map_location="cuda:1",weights_only=False)
        
        # SHAP 解释过程
shap = SHAPExplainer(
            input_data=test_input_data, 
            design_matrix=test_input_sign, 
            model=model,
            device="cuda:1"
        )

shap.explain(
            output_dir="/home/zxl/hdd/cellfate/model2", 
            iteration=0
   
        )
        



/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAPExplainer initialized.
torch.Size([1091, 7057])
torch.Size([8624, 7057])
[ 407  408  409 ... 1495 1496 1497]
当前时间: 2025-06-11 20:31:06.132816
torch.Size([8624, 7057])
torch.Size([1091, 7057])


Process SpawnProcess-5:
Process SpawnProcess-3:
Process SpawnProcess-4:
Traceback (most recent call last):
  File "/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
KeyboardInterrupt
  File "/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.r

In [ ]:
import pandas as pd

# 读取 CSV 文件
file_path = '/home/zxl/hdd/cellfate/data/subtype_primary_tpm_441.csv'
df = pd.read_csv(file_path)

# 去掉 sample 列中的双引号
df['sample'] = df['sample'].str.replace('"', '')

# 将修改后的数据保存回原文件
df.to_csv(file_path, index=False)
    

In [1]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer

test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_441.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_441.csv")
import gc


        # 加载模型到 GPU
model_path = f'/home/zxl/hdd/cellfate/model4/binn_model0_tpm_0.001_32.pth'
model = torch.load(model_path, map_location="cuda:1",weights_only=False)
        
        # SHAP 解释过程
shap = SHAPExplainer(
            input_data=test_input_data, 
            design_matrix=test_input_sign, 
            model=model,
            device="cuda:1"
        )

shap.explain_cell(
            output_dir="/home/zxl/hdd/cellfate/model2", 
            iteration=0

        )

/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAPExplainer initialized.
当前时间: 2025-06-10 14:57:32.909650
torch.Size([441, 688])


/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/shap/explainers/_deep/deep_pytorch.py:243: UserWarning: unrecognized nn.Module: Identity
  warnings.warn(f'unrecognized nn.Module: {module_type}')


当前时间: 2025-06-10 15:01:11.821182
SHAP dictionary saved to /home/zxl/hdd/cellfate/model2/shap_cell_iter0.csv


{'features': [['HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'HSC_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'CLP_B_cell_layers',
   'Pro_B_cell_layers',
   'Pro_B_cell_layers',
   'Pro_B_cell_layers',
   'Pro_B_cell_layers',
   'Pro_B_cell_layers',
   'Pro_B_cell_layers',
   'Pro_B_cell_layers',
   'Pro_B_cell_layers',
   'Pro_B_cell_layers',
   '

In [ ]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_441.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_441.csv")

import gc
for i in range(2, 10):
    # 确保从干净状态开始
    torch.cuda.empty_cache()  
    gc.collect()

    try:
        # 加载模型到 GPU
        model_path = f'/home/zxl/hdd/cellfate/model4/binn_model{i}_tpm_0.001_32.pth'
        model = torch.load(model_path, map_location="cuda:1",weights_only=False)  # 直接加载到 GPU
        
        # SHAP 解释过程
        shap = SHAPExplainer(
            input_data=test_input_data, 
            design_matrix=test_input_sign, 
            model=model
        )
        shap.explain_cell(
            output_dir="/home/zxl/hdd/cellfate/model1", 
            iteration=i
        )
        
        print(f"Completed iteration {i}")

    finally:
        # 显式释放资源
        if 'model' in locals():
            model.cpu()          # 将模型移出 GPU
            del model            # 删除模型引用
        if 'shap' in locals():
            del shap             # 删除 SHAP 解释器引用
        
        # 强制清理显存
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_441.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_441.csv")


import torch
from torch import cuda
import gc

for i in range(1, 10):
    # 确保从干净状态开始
    torch.cuda.empty_cache()  
    gc.collect()

    try:
        # 加载模型到 GPU
        model_path = f'/home/zxl/hdd/cellfate/model4/binn_model{i}_tpm_0.001_32.pth'
        model = torch.load(model_path, map_location="cuda:1",weights_only=False)  # 直接加载到 GPU
        
        # SHAP 解释过程
        shap = SHAPExplainer(
            input_data=test_input_data, 
            design_matrix=test_input_sign, 
            model=model
        )
        shap.explain(
            output_dir="/home/zxl/hdd/cellfate/model1", 
            iteration=i,
            fold="iteration"

        )
        
        print(f"Completed iteration {i}")

    finally:
        # 显式释放资源
        if 'model' in locals():
            model.cpu()          # 将模型移出 GPU
            del model            # 删除模型引用
        if 'shap' in locals():
            del shap             # 删除 SHAP 解释器引用
        
        # 强制清理显存
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
from binn.plot import complete_sankey
import os
import pandas as pd
from tqdm import tqdm  # 进度条支持
from binn.plot import complete_sankey

def batch_generate_svg(input_dir, output_dir):
    # 创建输出目录[3,4](@ref)
    os.makedirs(output_dir, exist_ok=True)
    
    # 获取所有CSV文件[6,7](@ref)
    csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]
    
    # 带进度条的批量处理[7](@ref)
    for csv_file in tqdm(csv_files, desc="生成SVG进度"):
        try:
            # 构建完整路径[3](@ref)
            input_path = os.path.join(input_dir, csv_file)
            output_path = os.path.join(output_dir, f"{os.path.splitext(csv_file)[0]}.svg")
            
            # 读取数据并生成图表[1](@ref)
            df = pd.read_csv(input_path)
            figure = complete_sankey(df)
            
            # 统一设置布局参数
            figure.update_layout(
                width=1200,
                height=800,
                margin=dict(l=50, r=50, t=50, b=50),
                font_size=12
            )
            
            # 保存为SVG[1,5](@ref)
            figure.write_image(
                output_path,
                format="svg",
                engine="kaleido"
            )
        except Exception as e:
            print(f"文件 {csv_file} 处理失败: {str(e)}")

# 使用示例
batch_generate_svg(
    "/home/zxl/hdd/cellfate/data/shap",  # 输入目录
    "/home/zxl/hdd/cellfate/svg_output"  # 输出目录
)



In [2]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
from binn.plot import complete_sankey
import os
import pandas as pd
from tqdm import tqdm  # 进度条支持
from binn.plot import complete_sankey            


df = pd.read_csv("/home/zxl/hdd/cellfate/data/shap/des_agg.csv")
figure = complete_sankey(df)

            # 统一设置布局参数
figure.update_layout(
                width=1200,
                height=800,
                margin=dict(l=50, r=50, t=50, b=50),
                font_size=12
            )
            
            # 保存为SVG[1,5](@ref)
figure.write_image(
                "/home/zxl/hdd/cellfate/svg_output/des_agg.svg",
                format="svg",
                engine="kaleido"
            )

KeyError: 'target.name'

In [ ]:
figure.write_image("/home/zxl/hdd/cellfate/result/plotly_fig.svg",format="svg") 

In [2]:
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score
import pandas as pd 
import numpy as np
import csv
#prediction_results_TCGA = pd.read_csv('/home/zxl/hdd/cellfate/result/prediction_result_TCGAtest_SVM.csv')
# prediction_results_TCGA["pred"] = prediction_results_TCGA["pred"]-1
# prediction_results_TCGA["V4"] = np.where(
#     prediction_results_TCGA["pred"] == prediction_results_TCGA["group"], 
#     1, 
#     0
# )
prediction_results_TCGA1165 = pd.read_csv('/home/zxl/hdd/lunwen数据/cellfate/result/prediction_results_TCGA720_XGBOOST.csv')
prediction_results_bcgsc = pd.read_csv('/home/zxl/hdd/lunwen数据/cellfate/result/TCGA720_prediction_BPformer.csv')
prediction_results_ICGC = pd.read_csv('/home/zxl/hdd/lunwen数据/cellfate/result/prediction_result_ICGC_SVM.csv')
prediction_results_MET500 = pd.read_csv('/home/zxl/hdd/lunwen数据/cellfate/result/prediction_result_MET500_SVM.csv')
def eval(pred):
    Y_pre_label_change = pred['pred'] 
    Y_true_label = pred['group'] 
    ture = (Y_pre_label_change == Y_true_label).astype(int)
    print('ACC_score',sum(ture==1)/len(ture))
    print('f1_score', f1_score(Y_pre_label_change, Y_true_label,average='weighted'))    
    print('precision_score', precision_score(Y_pre_label_change, Y_true_label, average='weighted', zero_division=0))
    print('recall_score', recall_score(Y_pre_label_change, Y_true_label, average='weighted', zero_division=0))




#eval(prediction_results_TCGA)
print('*'*100)
eval(prediction_results_bcgsc)
print('*'*100)
#eval(prediction_results_bcgsc)
print('*'*100)
#eval(prediction_results_ICGC)
print('*'*100)
#eval(prediction_results_MET500)




****************************************************************************************************
ACC_score 0.9569444444444445
f1_score 0.957071186184344
precision_score 0.9625492326731736
recall_score 0.9569444444444445
****************************************************************************************************
****************************************************************************************************
****************************************************************************************************


In [15]:
import xgboost as xgb
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, train_test_split, StratifiedShuffleSplit
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_100.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_100.csv")

def _fit_data_matrix_to_network_input(
        data_matrix: pd.DataFrame, features, feature_column="Gene"
    ) -> pd.DataFrame:
        nr_features_in_matrix = len(data_matrix.index)
        if len(features) > nr_features_in_matrix:
            features_df = pd.DataFrame(features, columns=[feature_column])
            data_matrix = data_matrix.merge(features_df, how="right", on=feature_column)
        if len(features) > 0:
            data_matrix.set_index(feature_column, inplace=True)
            features = list(features)
            data_matrix = data_matrix.loc[features]
        return data_matrix ,features
def _generate_k_folds(
        
        data_matrix: pd.DataFrame,
        design_matrix: pd.DataFrame,
        groups=list(range(1, 25)),  # 明确指定1-32类
        n_folds=3,
        test_size=0.2,  # 测试集比例15%
        val_size=0.2,   # 验证集比例15%
        random_state=42
    ):
        y = []
        dfs = []
        for i, group in enumerate(groups):
            group_samples = design_matrix[design_matrix["group"] == group]["sample"].values
            df_group = data_matrix[group_samples].T  # 转置为 (样本数, 特征数)
            dfs.append(df_group)
            y += [group-1 for _ in group_samples]    # 标签从0开始（0对应group 1，31对应group 32）

        y = np.array(y)
        X = pd.concat(dfs).fillna(0).to_numpy()
        #X = preprocessing.StandardScaler().fit_transform(X)

        splits = []
        skf_outer = StratifiedShuffleSplit(n_splits=n_folds, test_size=test_size, random_state=random_state)
        
        # 计算验证集在训练验证集中的相对比例
        val_size_relative = val_size / (1 - test_size)
        
        # 外层循环：将数据分为训练+验证集（85%）和测试集（15%）
        for train_val_index, test_index in skf_outer.split(X, y):
            X_train_val, X_test = X[train_val_index], X[test_index]
            y_train_val, y_test = y[train_val_index], y[test_index]
            
            # 内层循环：将训练+验证集分为训练集（70%）和验证集（15%）
            skf_inner = StratifiedShuffleSplit(n_splits=1, test_size=val_size_relative, random_state=random_state)
            for train_index, val_index in skf_inner.split(X_train_val, y_train_val):
                X_train, X_val = X_train_val[train_index], X_train_val[val_index]
                y_train, y_val = y_train_val[train_index], y_train_val[val_index]
                
                # 验证比例是否为70:15:15
                total = len(X)

                
                # 验证所有类别存在
  
                
                splits.append((X_train, y_train, X_val, y_val, X_test, y_test))
        
        return splits
import pickle
with open('/home/zxl/hdd/cellfate/data/Gene_and_network1.pkl', 'rb') as file:
    data = pickle.load(file)
gene_list = data["gene_list"]
# 加载示例数据集（威斯康星州乳腺癌数据集）

fitted_input_data ,feature= _fit_data_matrix_to_network_input(
                test_input_data.reset_index(),
                features=gene_list,
                feature_column="Gene"
            )
   
            
            
splits = _generate_k_folds(
                fitted_input_data, design_matrix=test_input_sign,n_folds=3, test_size= 0.2, val_size=0.2
            )
# 划分训练集和测试集
for split in splits:
    X_train, y_train, X_val, y_val, X_test, y_test = split
import numpy as np

print("唯一标签值:", np.unique(y_train))
print("最小值:", np.min(y_train), "最大值:", np.max(y_train))
y_train = y_train.astype(int)
y_test = y_test.astype(int)
# 转换为DMatrix格式（XGBoost高性能数据格式）
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# 定义基础参数
params = {
    'objective': 'multi:softmax',
    'num_class': len(np.unique(y_train)),          # 类别数量
    'eval_metric': 'mlogloss',
    'eta': 0.1,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42,
    'nthread': 4
}

# 训练模型（early_stopping需要验证集）
evals = [(dtrain, 'train'), (dtest, 'eval')]
model = xgb.train(
    params,
    dtrain,
    num_boost_round=100,  # 最大迭代次数
    evals=evals,
    early_stopping_rounds=50,  # 早停轮数
    verbose_eval=50            # 每50轮显示一次日志
)

# 预测测试集
y_pred_prob = model.predict(dtest)
y_pred = np.round(y_pred_prob)  # 概率转类别

# 评估模型
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")


# 保存模型
model.save_model('xgboost_classifier.model')

# 加载模型
loaded_model = xgb.Booster()
loaded_model.load_model('xgboost_classifier.model') 

唯一标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
最小值: 0 最大值: 23
[0]	train-mlogloss:2.29873	eval-mlogloss:2.36982
[50]	train-mlogloss:0.03045	eval-mlogloss:0.21261
[99]	train-mlogloss:0.00541	eval-mlogloss:0.15690

Confusion Matrix:
[[ 75   0   1   0   0   0   4   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   1]
 [  0 215   0   0   0   0   1   0   0   0   0   0   0   0   0   0   0   2
    0   0   0   0   0   0]
 [  1   0  59   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   1]
 [  0   1   0  54   0   0   0   0   0   0   0   0   1   0   1   0   0   0
    0   0   0   0   0   0]
 [  1   0   0   0  25   0   2   0   0   0   0   0   0   0   0   0   0   0
    0   8   0   0   0   0]
 [  0   0   0   0   0  28   0   0   0   1   0   0   0   0   0   0   0   1
    0   0   0   0   0   0]
 [  0   0   0   0   0   0 100   0   0   0   0   0   3   0   0   0   0   1
    0   0   0   0   0   0]
 [  0   0   0   0   0   0   0

/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [15:30:43] WARNING: /croot/xgboost-split_1724073744422/work/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


In [2]:
import xgboost as xgb
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, train_test_split, StratifiedShuffleSplit
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
def predict_and_evaluate(model, X, y_true):
    
    """
    生成预测概率和预测结果的表格
    参数:
        model: 训练好的PyTorch模型
        X: 输入数据 (Tensor)
        y_true: 真实标签 (Tensor)
    返回:
        DataFrame: 包含两列（预测概率，预测是否正确）的表格
    """
    y_pred_prob = model.predict(X)
    y_pred = np.round(y_pred_prob)  # 概率转类别
        

    
        # 转换为numpy数组
   
    pred_labels = y_pred
    y_true_np = y_true
        
        # 生成表格
    df = pd.DataFrame({
          
            "预测是否正确": (pred_labels == y_true_np).astype(int),
            "哪一类预测正确": y_true_np,               # 显示真实类别
            "预测类别":pred_labels 
        })
        
    return df
def predict_proba(model, X):
    X=torch.Tensor(X,dtype=torch.float32)
    with torch.no_grad():

        logits = model(X)
        probabilities = torch.softmax(logits, dim=1)
        return probabilities
def _fit_data_matrix_to_network_input(
    data_matrix: pd.DataFrame, feature, feature_column="Gene", 
        ) -> pd.DataFrame:
        nr_features_in_matrix = len(data_matrix.index)
        if len(feature) > nr_features_in_matrix:
            features_df = pd.DataFrame(feature, columns=[feature_column])
            data_matrix = data_matrix.merge(features_df, how="right", on=feature_column)
        if len(feature) > 0:
            data_matrix.set_index(feature_column, inplace=True)
            feature = list(feature)
      
            data_matrix = data_matrix.loc[feature]
        return data_matrix
def _generate_k_folds(
        data_matrix: pd.DataFrame,
        design_matrix: pd.DataFrame,
        groups=list(range(1, 25)),  # 明确指定 1 - 32 类
        n_folds=3,
        test_size=0.1,  # 测试集比例
        random_state=42
):
    y = []
    dfs = []
    sample_names = []  # 新增：用于记录样本名字
    for i, group in enumerate(groups):
        group_samples = design_matrix[design_matrix["group"] == group]["sample"].values
        df_group = data_matrix[group_samples].T  # 转置为 (样本数, 特征数)
        dfs.append(df_group)
        y += [group-1  for _ in group_samples]  # 标签从 0 开始（0 对应 group 1，31 对应 group 32）
        sample_names.extend(group_samples)  # 新增：记录样本名字

    y = np.array(y)
    X = pd.concat(dfs).fillna(0).to_numpy()
    # X = preprocessing.StandardScaler().fit_transform(X)
    return X, y, sample_names  # 新增：返回样本名字
# 示例用法

if __name__ == "__main__":
    loaded_model = xgb.Booster()
    loaded_model.load_model('xgboost_classifier.model') 
    input_data = pd.read_csv("/home/zxl/hdd/lunwen数据/cellfate/data/TCGA_720_exp.csv")
    import pickle
    with open('/home/zxl/hdd/lunwen数据/cellfate/data/Gene_and_network1.pkl', 'rb') as file:
        data = pickle.load(file)
    features = data["gene_list"]
    y_true = pd.read_csv("/home/zxl/hdd/lunwen数据/cellfate/data/TCGA_720_group.csv")
    fitted_input_data = _fit_data_matrix_to_network_input(
                input_data.reset_index(),
                
                feature_column="Gene",
                feature = features
            )
    X, y, sample_names = _generate_k_folds(
                fitted_input_data, design_matrix=y_true,n_folds=3
            )
    test_data = X
    tensor= y
        # 确保 y 的形状是 (num_samples,)
    dtest = xgb.DMatrix(test_data, label=y)



    
    # 生成结果表格
    result_df = predict_and_evaluate(loaded_model, dtest, y)
    
    # 查看前5行
    print(result_df.head())
    result_df.to_csv("/home/zxl/hdd/lunwen数据/cellfate/result/prediction_results_TCGA720_XGBOOST.csv", index=False)




   预测是否正确  哪一类预测正确  预测类别
0       1        0   0.0
1       1        0   0.0
2       1        0   0.0
3       1        0   0.0
4       1        0   0.0


In [1]:
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
import os
from docs.util_for_examples import fit_data_matrix_to_network_input
from binn.explainer import BINNExplainer
from binn.based_cell_train import based_cell_train
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, train_test_split
import pickle
from datetime import datetime
from collections import defaultdict
cell="TCGA"
def _generate_k_folds(
        data_matrix: pd.DataFrame,
        design_matrix: pd.DataFrame
):
 
    dfs = []
    sample_names = []  # 新增：用于记录样本名字

    group_samples = design_matrix["sample"].values
    df_group = data_matrix[group_samples].T  # 转置为 (样本数, 特征数)
    dfs.append(df_group)
    sample_names.extend(group_samples)  # 新增：记录样本名字
    X = pd.concat(dfs).fillna(0).to_numpy()
    # X = preprocessing.StandardScaler().fit_transform(X)
    return X,  sample_names  # 新增：返回样本名字


def _fit_data_matrix_to_network_input(
         data_matrix: pd.DataFrame, features, feature_column="Gene"
    ) -> pd.DataFrame:
        nr_features_in_matrix = len(data_matrix.index)
        if len(features) > nr_features_in_matrix:
            features_df = pd.DataFrame(features, columns=[feature_column])
            data_matrix = data_matrix.merge(features_df, how="right", on=feature_column)
        if len(features) > 0:
            data_matrix.set_index(feature_column, inplace=True)
            data_matrix = data_matrix.loc[features]
        return data_matrix

test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_441.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_441.csv",sep=',')
test_pathways = pd.read_csv("/home/zxl/hdd/lsy/data/BP_5.csv")
test_translation = pd.read_csv("/home/zxl/hdd/lsy/data/BP_Gene_new.csv")
with open('/home/zxl/hdd/cellfate/data/Gene_and_network1.pkl', 'rb') as file:
    data = pickle.load(file)
gene_list = data["gene_list"]

gene_set = set(gene_list.tolist()) 
network = Network(
    input_data=test_input_data,
    pathways=test_pathways,
    mapping=test_translation,
    input_data_column="Gene",
    source_column="source",
    target_column="target"
)


binn = BINN(
    network=network,
    activation = "tanh",
    activation_final = "sigmoid",
    connectivity_matrices_list = data,
    dropout=0.2,
    validate=False,
    device="cuda:1",
    learning_rate=0.001,
) 


input_data = test_input_data
design_matrix = test_input_sign

fitted_input_data = _fit_data_matrix_to_network_input(
                input_data.reset_index(),
                features=data["gene_list"],
                feature_column="Gene"
            )

X, y = _generate_k_folds(
                fitted_input_data, design_matrix=design_matrix)

test_data = torch.Tensor(X)
background_data = torch.Tensor(X)
background_data = test_data
test_data = test_data
y=y

filename="/home/zxl/hdd/cellfate/model4/binn_model0_tpm_0.001_32.pth"

    
model = torch.load(filename, map_location="cuda:1", weights_only=False)
explainer =BINNExplainer(model)

current_time = datetime.now()
print("当前时间:", current_time)
shap_dict = explainer._explain_cell_layer(test_data, y, background_data)
current_time = datetime.now()
print("当前时间:", current_time)

feature_dict = {
            "name": [],
            "type": [],
    }
samples = y

for sample in samples:
    feature_dict[sample] = []  # 动态添加新键并初始化空列表
    


feature_id_mapping = {}
feature_id = 0
feature_id_mapping["root"] = feature_id
for layer_features in shap_dict["features"]:
    for feature in layer_features:
        feature_id += 1
        feature_id_mapping[feature] = feature_id

curr_layer = 0

for sv, features in zip(
                    shap_dict["shap_values"], shap_dict["features"]
):
    sv = np.asarray(sv)
            


        
    for feature in range(sv.shape[1]):
        n_classes = sv.shape[2]
        for curr_class in range(n_classes):

            feature_dict["name"].append(features[feature])

            feature_dict["type"].append(curr_class)             
                
            sample_index = 0
            for sample in samples:
                    
                feature_dict[sample].append(sv[sample_index][feature][curr_class])
                sample_index = sample_index + 1
                    

    curr_layer += 1
    print(curr_layer)
df = pd.DataFrame(data=feature_dict)
output_file = '/home/zxl/hdd/cellfate/model2/shap_TCGA_0.csv'
df.to_csv(output_file, index=False)

/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zxl/hdd/cellfate/binn/binn.py:214: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.loss = nn.CrossEntropyLoss(weight=torch.tensor(self.weight, device=device))



BINN is on the device: cuda:1
当前时间: 2025-06-10 15:41:34.197870
torch.Size([441, 688])


/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/shap/explainers/_deep/deep_pytorch.py:243: UserWarning: unrecognized nn.Module: Identity
  warnings.warn(f'unrecognized nn.Module: {module_type}')


当前时间: 2025-06-10 15:45:02.911230
1


In [1]:
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
import os
from docs.util_for_examples import fit_data_matrix_to_network_input
from binn.explainer import BINNExplainer
from binn.based_cell_train import based_cell_train
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, train_test_split
import pickle
from datetime import datetime
from collections import defaultdict
import dill


target_key = ("HSC_CD8_cell_Tem_layers","CLP_CD8_cell_Tem_layers","Pre_CD8_cell_Tem_layers","Pro_CD8_cell_Tem_layers","naive_CD8_cell_Tem_layers","Teff_CD8_cell_Tem_layers")
def _generate_k_folds(
    data_matrix: pd.DataFrame,
    design_matrix: pd.DataFrame,
    groups=list(range(1, 25)),  # 明确指定1-32类
    n_folds=3,
    test_size=0.15,  # 测试集比例15%
    val_size=0.15,   # 验证集比例15%
    random_state=42
):
    y = []
    dfs = []
    for i, group in enumerate(groups):
        group_samples = design_matrix[design_matrix["group"] == group]["sample"].values
        df_group = data_matrix[group_samples].T  # 转置为 (样本数, 特征数)

        dfs.append(df_group)
        y += [group-1 for _ in group_samples]    # 标签从0开始（0对应group 1，31对应group 32）
        if group==13:
            y_test = group_samples
            

    test_data = data_matrix[group_samples].T.fillna(0).to_numpy()

    y = np.array(y)
    X = pd.concat(dfs).fillna(0).to_numpy()
    #X = preprocessing.StandardScaler().fit_transform(X)
    return X, y,test_data,y_test


def _fit_data_matrix_to_network_input(
         data_matrix: pd.DataFrame, features, feature_column="Gene"
    ) -> pd.DataFrame:
        nr_features_in_matrix = len(data_matrix.index)
        if len(features) > nr_features_in_matrix:
            features_df = pd.DataFrame(features, columns=[feature_column])
            data_matrix = data_matrix.merge(features_df, how="right", on=feature_column)
        if len(features) > 0:
            data_matrix.set_index(feature_column, inplace=True)
            data_matrix = data_matrix.loc[features]
        return data_matrix

test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_441.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_441.csv",sep=',')
test_pathways = pd.read_csv("/home/zxl/hdd/lsy/data/BP_5.csv")
test_translation = pd.read_csv("/home/zxl/hdd/lsy/data/BP_Gene_new.csv")
with open('/home/zxl/hdd/cellfate/data/Gene_and_network1.pkl', 'rb') as file:
    data = pickle.load(file)
gene_list = data["gene_list"]

network = Network(
    input_data=test_input_data,
    pathways=test_pathways,
    mapping=test_translation,
    input_data_column="Gene",
    source_column="source",
    target_column="target"
)


binn = BINN(
    network=network,
    activation = "tanh",
    activation_final = "sigmoid",
    connectivity_matrices_list = data,
    dropout=0.2,
    validate=False,
    device="cuda:1",
    learning_rate=0.001,
) 


input_data = test_input_data
design_matrix = test_input_sign

fitted_input_data = _fit_data_matrix_to_network_input(
                input_data.reset_index(),
                features=data["gene_list"],
                feature_column="Gene"
            )

X, y, test_data, y_test= _generate_k_folds(
                data_matrix = fitted_input_data, design_matrix=design_matrix,n_folds=3
            )
        

background_data = torch.Tensor(X)


# variables_to_save = { 'y_test': y_test,'fitted_input_data': fitted_input_data,'y':y}
# filename = '/home/zxl/hdd/cellfate/model4/y_test_LUSC.pkl'
# with open(filename, 'wb') as f:
#     dill.dump(variables_to_save, f)
# with open('/home/zxl/hdd/cellfate/model4/y_test_LUAD.pkl', 'rb') as file:
#     data = pickle.load(file)
# y_test = data["y_test"]
# fitted_input_data = data["fitted_input_data"]
# y = data["y"]
# print(y_test)


# df_group1 = fitted_input_data[y_test[:100]].T.fillna(0).to_numpy()
# df_group1 = torch.Tensor(df_group1)
# df_group2 = fitted_input_data[y_test[100:500]].T.fillna(0).to_numpy()
# df_group2 = torch.Tensor(df_group2)
# df_group3 = fitted_input_data[y_test].T.fillna(0).to_numpy()
# df_group3 = torch.Tensor(df_group3)




def get_connectivity_matrices_list():
    connectivity_matrices_list = model.B_cell_connectivity_matrices[:4]+ model.CD8_cell_Tcm_connectivity_matrices[:4]+ model.CD8_cell_Tem_connectivity_matrices[:4]+ model.CD4_cell_Th1_connectivity_matrices[:4]+model.CD4_cell_Th2_connectivity_matrices[:4]+ model.CD4_cell_Th17_connectivity_matrices[:4]+model.CD4_cell_Tfh_connectivity_matrices[:4]+ model.CD4_cell_Treg_connectivity_matrices[:4]+ model.B_cell_connectivity_matrices[6:10]+model.B_cell_connectivity_matrices[12:16]+ model.B_cell_connectivity_matrices[18:22]+model.B_cell_connectivity_matrices[24:28]+model.B_cell_connectivity_matrices[30:34]+model.CD8_cell_Tcm_connectivity_matrices[6:10]+model.CD8_cell_Tcm_connectivity_matrices[12:16]+  model.CD8_cell_Tcm_connectivity_matrices[18:22]+ model.CD8_cell_Tcm_connectivity_matrices[24:28]+ model.CD8_cell_Tcm_connectivity_matrices[30:34]+model.CD8_cell_Tem_connectivity_matrices[6:10]+  model.CD8_cell_Tem_connectivity_matrices[12:16]+  model.CD8_cell_Tem_connectivity_matrices[18:22]+ model.CD8_cell_Tem_connectivity_matrices[24:28]+ model.CD8_cell_Tem_connectivity_matrices[30:34]+model.CD4_cell_Th1_connectivity_matrices[6:10]+ model.CD4_cell_Th1_connectivity_matrices[12:16]+ model.CD4_cell_Th1_connectivity_matrices[18:22]+ model.CD4_cell_Th1_connectivity_matrices[24:28]+ model.CD4_cell_Th2_connectivity_matrices[6:10]+ model.CD4_cell_Th2_connectivity_matrices[12:16]+ model.CD4_cell_Th2_connectivity_matrices[18:22]+ model.CD4_cell_Th2_connectivity_matrices[24:28]+model.CD4_cell_Th17_connectivity_matrices[6:10]+ model.CD4_cell_Th17_connectivity_matrices[12:16]+ model.CD4_cell_Th17_connectivity_matrices[18:22]+ model.CD4_cell_Th17_connectivity_matrices[24:28]+model.CD4_cell_Tfh_connectivity_matrices[6:10]+ model.CD4_cell_Tfh_connectivity_matrices[12:16]+ model.CD4_cell_Tfh_connectivity_matrices[18:22]+ model.CD4_cell_Tfh_connectivity_matrices[24:28]+model.CD4_cell_Treg_connectivity_matrices[6:10]+ model.CD4_cell_Treg_connectivity_matrices[12:16]+ model.CD4_cell_Treg_connectivity_matrices[18:22]+ model.CD4_cell_Treg_connectivity_matrices[24:28]
    return connectivity_matrices_list

filename="/home/zxl/hdd/cellfate/model4/binn_model0_tpm_0.001_32.pth"

model = torch.load(filename, map_location="cuda:1", weights_only=False)
explainer =BINNExplainer(model)
device = "cuda:1"
# current_time = datetime.now()
# print("当前时间:", current_time)

# shap_dict, shap_dict_cell = explainer._explain_layer1(background_data,df_group3,device)
# current_time = datetime.now()
# print("当前时间:", current_time)


# variables_to_save = { 'shap_dict': shap_dict,"shap_dict_cell":shap_dict_cell,"y_test1":y_test}
# filename = '/home/zxl/hdd/cellfate/model4/shap_dict_LUAD.pkl'
# with open(filename, 'wb') as f:
#     dill.dump(variables_to_save, f)


# current_time = datetime.now()
# print("当前时间:", current_time)

# shap_dict, shap_dict_cell = explainer._explain_layer1(background_data,df_group2, y_z_2,device)
# current_time = datetime.now()
# print("当前时间:", current_time)
# variables_to_save = { 'shap_dict': shap_dict,"y_test2":y_test[380:760]}
# filename = '/home/zxl/hdd/cellfate/model4/shap_dict_2.pkl'
# with open(filename, 'wb') as f:
#     dill.dump(variables_to_save, f)


# current_time = datetime.now()
# print("当前时间:", current_time)

# shap_dict, shap_dict_cell = explainer._explain_layer1(background_data,df_group3, y_z_3,device)
# current_time = datetime.now()
# print("当前时间:", current_time)
# variables_to_save = { 'shap_dict': shap_dict,"y_test3":y_test[760:]}
# filename = '/home/zxl/hdd/cellfate/model4/shap_dict_3.pkl'
# with open(filename, 'wb') as f:
#     dill.dump(variables_to_save, f)


with open('/home/zxl/hdd/cellfate/model4/shap_dict_LUAD.pkl', 'rb') as file:
    data = pickle.load(file)
shap_dict = data["shap_dict"]
y_test = data["y_test1"]
feature_dict = {
            "name": [],
            "source name": [],
            "target name": [],
      
            "type": [],
            "source layer":[],
            "target layer": [],                  
    }

samples = y_test
for sample in samples:
    feature_dict[sample] = []  # 动态添加新键并初始化空列表
    


feature_id_mapping = {}
feature_id = 0
feature_id_mapping["root"] = feature_id
for layer_features in shap_dict["features"]:
    for feature in layer_features:
        feature_id += 1
        feature_id_mapping[feature] = feature_id

curr_layer = 0

connectivity_matrices_list = get_connectivity_matrices_list()
feature_id_mapping = {}
feature_id = 0
feature_id_mapping["root"] = feature_id
for layer_features in shap_dict["features"]:
    for feature in layer_features:
        feature_id += 1
        feature_id_mapping[feature] = feature_id

        curr_layer = 0

values_cell = np.asarray(shap_dict["shap_values"][-1])  #每个细胞对最终结果的贡献值
values_cell = abs(values_cell)
values_cell_mean = np.mean(values_cell, axis=0)
element = []
first_elements = [vector[0] for vector in shap_dict["features"][:-1]]
index_dict  = defaultdict(list)

for index, item in enumerate(shap_dict["features"][-1]):
    index_dict[item].append(index)
    element.append(item) 
element = set(element)
prefix_indices = {prefix: [] for prefix in element}

# 遍历列表，找到每个元素的前缀，并将索引添加到对应的前缀组中
for index, item in enumerate(first_elements):
    for prefix in element:
        if item.startswith(prefix):
            prefix_indices[prefix].append(index)

# 输出结果
for prefix, indices in prefix_indices.items():
    print(f"Prefix '{prefix}' found at indices: {indices}")

merged_dict = {}

# 合并每一层
for key in prefix_indices:
    # 获取每个层的列表
    layers = prefix_indices[key]
    values = index_dict.get(key, [])
    
    # 将对应的层和其值合并成一个元组
    merged_dict[key] = [layers, values]



selected_data = {
    key: merged_dict[key] 
    for key in target_key 
    if key in merged_dict
}
merged_dict=selected_data        
print(merged_dict)
for name, (first, second) in merged_dict.items(): 


    shap_dict_items = {"features":[], "shap_values":[]}
    connectivity_matrices_list_items = []
    for idx in first:
        shap_dict_items["shap_values"].append(shap_dict["shap_values"][idx])
        shap_dict_items["features"].append(shap_dict["features"][idx])
        connectivity_matrices_list_items.append(connectivity_matrices_list[idx])
    result_second = values_cell_mean[second]  #短的部分乘的是短shap值，长的乘以的是全部的，短代表就是哪个阶段（例如非成熟b细胞到成熟b细胞）对最终的影响。长的代表的是整个阶段（例如B细胞发育的整个阶段）对最终的影响
    curr_layer = 0


    for sv, features, cm  in zip(
            shap_dict_items["shap_values"], shap_dict_items["features"], connectivity_matrices_list_items
    ):
        sv = np.asarray(sv)
                
        sv_mean_final =  sv @ result_second   
        for feature in range(sv_mean_final.shape[1]):

            n_classes = sv_mean_final.shape[2]
            modified_string = features[feature].replace(name+'_', '')
            connections = cm[cm.index == modified_string]
            connections = connections.loc[
                    :, (connections != 0).any(axis=0)
            ]  # get targets and append to target
            for target in connections:
                for curr_class in range(n_classes):
                    if curr_class != 11: 
                        continue 
                    feature_dict["name"].append(name)
                    feature_dict["source name"].append(features[feature])
                    feature_dict["target name"].append(name + "_" + target)
                    feature_dict["type"].append(curr_class)
                    feature_dict["source layer"].append(curr_layer)
                    feature_dict["target layer"].append(curr_layer + 1)
                    sample_index = 0
                    for sample in samples:
                        
                        feature_dict[sample].append(sv_mean_final[sample_index][feature][curr_class])
                        sample_index = sample_index + 1    
        curr_layer += 1
            
            
    df = pd.DataFrame(data=feature_dict)
    output_file = '/home/zxl/hdd/cellfate/model2/shap_GO_LUAD_Tem.csv'.format()
    df.to_csv(output_file, index=False)






/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zxl/hdd/cellfate/binn/binn.py:214: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.loss = nn.CrossEntropyLoss(weight=torch.tensor(self.weight, device=device))



BINN is on the device: cuda:1
Prefix 'Pre_CD8_cell_Tcm_layers' found at indices: [60, 61, 62, 63]
Prefix 'CLP_CD4_cell_Tfh_layers' found at indices: [140, 141, 142, 143]
Prefix 'Pre_CD4_cell_Treg_layers' found at indices: [164, 165, 166, 167]
Prefix 'Pre_B_cell_layers' found at indices: [40, 41, 42, 43]
Prefix 'Pre_CD4_cell_Th1_layers' found at indices: [100, 101, 102, 103]
Prefix 'HSC_CD4_cell_Treg_layers' found at indices: [28, 29, 30, 31]
Prefix 'Pro_CD4_cell_Th1_layers' found at indices: [96, 97, 98, 99]
Prefix 'CLP_CD8_cell_Tcm_layers' found at indices: [52, 53, 54, 55]
Prefix 'CLP_B_cell_layers' found at indices: [32, 33, 34, 35]
Prefix 'mature_B_cell_layers' found at indices: [48, 49, 50, 51]
Prefix 'naive_CD8_cell_Tem_layers' found at indices: [84, 85, 86, 87]
Prefix 'naive_CD4_cell_Th17_layers' found at indices: [136, 137, 138, 139]
Prefix 'CLP_CD4_cell_Th1_layers' found at indices: [92, 93, 94, 95]
Prefix 'Pro_CD4_cell_Treg_layers' found at indices: [160, 161, 162, 163]
Pref

In [1]:
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
import os
from docs.util_for_examples import fit_data_matrix_to_network_input
from binn.explainer import BINNExplainer
from binn.based_cell_train import based_cell_train
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, train_test_split
import pickle
from datetime import datetime
from collections import defaultdict
import dill


#target_key = ("HSC_B_cell_layers","CLP_B_cell_layers","Pre_B_cell_layers","Pro_B_cell_layers","immature_B_cell_layers","mature_B_cell_layers")
def get_connectivity_matrices_list():
    connectivity_matrices_list = model.B_cell_connectivity_matrices[:4]+ model.CD8_cell_Tcm_connectivity_matrices[:4]+ model.CD8_cell_Tem_connectivity_matrices[:4]+ model.CD4_cell_Th1_connectivity_matrices[:4]+model.CD4_cell_Th2_connectivity_matrices[:4]+ model.CD4_cell_Th17_connectivity_matrices[:4]+model.CD4_cell_Tfh_connectivity_matrices[:4]+ model.CD4_cell_Treg_connectivity_matrices[:4]+ model.B_cell_connectivity_matrices[6:10]+model.B_cell_connectivity_matrices[12:16]+ model.B_cell_connectivity_matrices[18:22]+model.B_cell_connectivity_matrices[24:28]+model.B_cell_connectivity_matrices[30:34]+model.CD8_cell_Tcm_connectivity_matrices[6:10]+model.CD8_cell_Tcm_connectivity_matrices[12:16]+  model.CD8_cell_Tcm_connectivity_matrices[18:22]+ model.CD8_cell_Tcm_connectivity_matrices[24:28]+ model.CD8_cell_Tcm_connectivity_matrices[30:34]+model.CD8_cell_Tem_connectivity_matrices[6:10]+  model.CD8_cell_Tem_connectivity_matrices[12:16]+  model.CD8_cell_Tem_connectivity_matrices[18:22]+ model.CD8_cell_Tem_connectivity_matrices[24:28]+ model.CD8_cell_Tem_connectivity_matrices[30:34]+model.CD4_cell_Th1_connectivity_matrices[6:10]+ model.CD4_cell_Th1_connectivity_matrices[12:16]+ model.CD4_cell_Th1_connectivity_matrices[18:22]+ model.CD4_cell_Th1_connectivity_matrices[24:28]+ model.CD4_cell_Th2_connectivity_matrices[6:10]+ model.CD4_cell_Th2_connectivity_matrices[12:16]+ model.CD4_cell_Th2_connectivity_matrices[18:22]+ model.CD4_cell_Th2_connectivity_matrices[24:28]+model.CD4_cell_Th17_connectivity_matrices[6:10]+ model.CD4_cell_Th17_connectivity_matrices[12:16]+ model.CD4_cell_Th17_connectivity_matrices[18:22]+ model.CD4_cell_Th17_connectivity_matrices[24:28]+model.CD4_cell_Tfh_connectivity_matrices[6:10]+ model.CD4_cell_Tfh_connectivity_matrices[12:16]+ model.CD4_cell_Tfh_connectivity_matrices[18:22]+ model.CD4_cell_Tfh_connectivity_matrices[24:28]+model.CD4_cell_Treg_connectivity_matrices[6:10]+ model.CD4_cell_Treg_connectivity_matrices[12:16]+ model.CD4_cell_Treg_connectivity_matrices[18:22]+ model.CD4_cell_Treg_connectivity_matrices[24:28]
    return connectivity_matrices_list

filename="/home/zxl/hdd/cellfate/model4/binn_model0_tpm_0.001_32.pth"

model = torch.load(filename, map_location="cuda:1", weights_only=False)
with open('/home/zxl/hdd/cellfate/model4/shap_dict_LUSC.pkl', 'rb') as file:
    data = pickle.load(file)

y_test = data["y_test1"]
shap_dict_cell = data["shap_dict_cell"]
shap_dict = shap_dict_cell

feature_dict = {
            "name": [],
            "type": [],
        
    }

samples = y_test

for sample in samples:
    feature_dict[sample] = []  # 动态添加新键并初始化空列表
    


feature_id_mapping = {}
feature_id = 0
feature_id_mapping["root"] = feature_id
for layer_features in shap_dict["features"]:
    for feature in layer_features:
        feature_id += 1
        feature_id_mapping[feature] = feature_id

curr_layer = 0

for sv, features in zip(
                    shap_dict["shap_values"], shap_dict["features"]
):
    sv = np.asarray(sv)
            


        
    for feature in range(sv.shape[1]):
        n_classes = sv.shape[2]
        for curr_class in range(n_classes):

            feature_dict["name"].append(features[feature])

            feature_dict["type"].append(curr_class)             
                
            sample_index = 0
            for sample in samples:
                    
                feature_dict[sample].append(sv[sample_index][feature][curr_class])
                sample_index = sample_index + 1
                    

    curr_layer += 1
    print(curr_layer)
df = pd.DataFrame(data=feature_dict)
output_file = '/home/zxl/hdd/cellfate/model2/shap_LUSC_cell.csv'
df.to_csv(output_file, index=False)

/home/zxl/.conda/envs/cellfate_5090/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1
